# Amazon Book Price Estimator
## Modelling & Fine-tuning

This notebook runs traditional ML baselines and fine-tunes a GPT model to predict book prices from text alone.

**Files needed:** `splits.pkl`, `evaluator.py`

### Please run the book_pricer_data.ipynb first to generate `splits.pkl` file.

The `evaluator.py` file is used to evaluate your results

**Approach:**
1. Traditional ML baselines - establish a target to beat
2. Fine-tune `gpt-4.1-nano` with bucket sampling experiments and compare with Traditional ML baseline result

**Price buckets for this dataset ($1.06 - $17.24):**
```
$1.00 - $2.50   cheap paperbacks
$2.50 - $4.00   standard paperbacks
$4.00 - $6.00   mid range
$6.00 - $10.00  premium
$10.00+         expensive / hardcovers
```

In [ ]:
import os
import json
import pickle
import random
import numpy as np
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer
from evaluator import evaluate
from xgboost import XGBModel

@dataclass
class Item:
    title: str
    price: float
    prompt: str

    def __repr__(self):
        return f"Item(price=${self.price:.2f}, title='{self.title[:50]}')"

print('Ready')

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
with open('splits.pkl', 'rb') as f:
    splits = pickle.load(f)

train = splits['train']
val   = splits['val']
test  = splits['test']

print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')

---
# Part 1 — Traditional ML Baselines

We will run five models ranging from naive to sophisticated.
The best result here is the target the fine-tuned LLM should beat.

In [ ]:
# Random Pricer
min_price = min(i.price for i in train)
max_price = max(i.price for i in train)

def random_pricer(item):
    return random.uniform(min_price, max_price)

random.seed(42)
evaluate(random_pricer, test)

In [ ]:
# Constant Pricer — always predict the training mean
training_mean = sum(i.price for i in train) / len(train)
print(f'Training mean: ${training_mean:.2f}')

def constant_pricer(item):
    return training_mean

evaluate(constant_pricer, test)

In [ ]:
# Vectorize prompts — shared by Linear Regression, Random Forest and XGBoost
train_docs   = [item.prompt for item in train]
train_prices = np.array([item.price for item in train])

vectorizer = CountVectorizer(max_features=500, stop_words='english')
X_train = vectorizer.fit_transform(train_docs)
print(f'Vectorized {len(train_docs):,} docs into {X_train.shape[1]} features')

In [ ]:
# Using Linear Regression
lr = LinearRegression()
lr.fit(X_train, train_prices)

def linear_regression_pricer(item):
    x = vectorizer.transform([item.prompt])
    return max(lr.predict(x)[0], 0)

evaluate(linear_regression_pricer, test)

In [ ]:
# Using Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, train_prices)

def random_forest_pricer(item):
    x = vectorizer.transform([item.prompt])
    return max(rf.predict(x)[0], 0)

evaluate(random_forest_pricer, test)

In [ ]:
# XGBoost — best traditional ML model
import xgboost as xgb

xgb_model = xgb.XGBRegressor(n_estimators=300, random_state=42, n_jobs=-1, learning_rate=0.1)
xgb_model.fit(X_train, train_prices)

def xgboost_pricer(item):
    x = vectorizer.transform([item.prompt])
    return max(xgb_model.predict(x)[0], 0)

evaluate(xgboost_pricer, test)

In [ ]:
# Fill in after running each model above, below is my own result
ml_results = {
    'Random pricer':     5.18,
    'Constant pricer':   2.10,
    'Linear Regression': 1.80,
    'Random Forest':     1.60,
    'XGBoost':           1.55,
}

best = min(ml_results.values())

print('TRADITIONAL ML RESULTS')
print('=' * 45)
for name, error in ml_results.items():
    flag = '  <- TARGET TO BEAT' if error == best else ''
    print(f'  {name:<25} ${error:.2f}{flag}')
print('=' * 45)

---
# Part 2 — Fine-tuning

We fine-tune `gpt-4.1-nano` on book descriptions and compare against the ML baseline above.

**Key idea — Bucket Sampling:**
Instead of picking training examples randomly, we force equal representation across price tiers.
The dataset is skewed toward cheap paperbacks — random selection gives the model almost no
exposure to expensive books. Bucket sampling fixes this.

**Run one experiment at a time:**
1. Run the upload cell
2. Run the fine-tune cell
3. Wait for the job at https://platform.openai.com/finetune
4. Run the eval cell
5. Move to the next experiment

In [ ]:
# def functions to be used by the book_pricer_models.ipynb

def messages_for(item):
    return [
        {"role": "user",      "content": item.prompt},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

def test_messages_for(item):
    return [{"role": "user", "content": item.prompt}]


def write_jsonl(items, filename):
    os.makedirs('jsonl', exist_ok=True)
    with open(filename, 'w') as f:
        for item in items:
            f.write(json.dumps({'messages': messages_for(item)}) + '\n')
    print(f'Wrote {len(items)} examples to {filename}')


def upload_jsonl(filename):
    with open(filename, 'rb') as f:
        return openai.files.create(file=f, purpose='fine-tune')


def get_model_name(job_id):
    job = openai.fine_tuning.jobs.retrieve(job_id)
    print(f'Status: {job.status}')
    if job.fine_tuned_model:
        print(f'Model: {job.fine_tuned_model}')
    else:
        print('Not ready yet — check https://platform.openai.com/finetune')
    return job.fine_tuned_model


def make_pricer(model_name):
    def pricer(item):
        response = openai.chat.completions.create(
            model=model_name,
            messages=test_messages_for(item),
            max_tokens=10
        )
        return response.choices[0].message.content
    return pricer


def bucket_sample(items, per_bucket, seed=42):
    """Pick per_bucket items evenly from each price tier."""
    price_buckets = {
        '$1.00-2.50':  [i for i in items if i.price < 2.50],
        '$2.50-4.00':  [i for i in items if 2.50 <= i.price < 4.00],
        '$4.00-6.00':  [i for i in items if 4.00 <= i.price < 6.00],
        '$6.00-10.00': [i for i in items if 6.00 <= i.price < 10.00],
        '$10.00+':     [i for i in items if i.price >= 10.00],
    }
    random.seed(seed)
    sampled = []
    print(f'Bucket sampling ({per_bucket} per bucket):')
    for name, bucket in price_buckets.items():
        picked = random.sample(bucket, min(per_bucket, len(bucket)))
        sampled.extend(picked)
        bar = chr(9608) * len(picked)
        print(f'  {name:14s} {len(picked):3d}  {bar}  (${min(i.price for i in picked):.2f}-${max(i.price for i in picked):.2f})')
    random.shuffle(sampled)
    print(f'Total: {len(sampled)}')
    return sampled


print('Helpers ready')

---
## Baseline — First 100 Examples

Random selection, 1 epoch. Establishes the starting reference for fine-tuning.

In [ ]:
baseline_train = train[:100]
baseline_val   = val[:50]

write_jsonl(baseline_train, 'jsonl/baseline_train.jsonl')
write_jsonl(baseline_val,   'jsonl/baseline_val.jsonl')

baseline_train_file = upload_jsonl('jsonl/baseline_train.jsonl')
baseline_val_file   = upload_jsonl('jsonl/baseline_val.jsonl')

baseline_job = openai.fine_tuning.jobs.create(
    training_file=baseline_train_file.id,
    validation_file=baseline_val_file.id,
    model='gpt-4.1-nano-2025-04-14',
    seed=42,
    hyperparameters={'n_epochs': 1, 'batch_size': 1},
    suffix='books-baseline'
)
baseline_job_id = baseline_job.id
print(f'Job started: {baseline_job_id}')
print('Check: https://platform.openai.com/finetune')

In [ ]:
# Run after job finishes
baseline_job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
baseline_model  = get_model_name(baseline_job_id)
evaluate(make_pricer(baseline_model), test)

---
## Experiment 1 — 100 Bucket-Sampled Examples

20 examples per price tier. Same total as baseline but balanced across cheap and expensive books.

In [ ]:
exp1_train = bucket_sample(train, per_bucket=20)
exp1_val   = val[:50]

write_jsonl(exp1_train, 'jsonl/exp1_train.jsonl')
write_jsonl(exp1_val,   'jsonl/exp1_val.jsonl')

exp1_train_file = upload_jsonl('jsonl/exp1_train.jsonl')
exp1_val_file   = upload_jsonl('jsonl/exp1_val.jsonl')

exp1_job = openai.fine_tuning.jobs.create(
    training_file=exp1_train_file.id,
    validation_file=exp1_val_file.id,
    model='gpt-4.1-nano-2025-04-14',
    seed=42,
    hyperparameters={'n_epochs': 1, 'batch_size': 1},
    suffix='books-bucket100'
)
exp1_job_id = exp1_job.id
print(f'Job started: {exp1_job_id}')
print('Check: https://platform.openai.com/finetune')

In [ ]:
# Run after job finishes
exp1_job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
exp1_model  = get_model_name(exp1_job_id)
evaluate(make_pricer(exp1_model), test)

---
## Experiment 2 — 200 Bucket-Sampled Examples

40 per tier. Double the training data, still balanced.

In [ ]:
exp2_train = bucket_sample(train, per_bucket=40)
exp2_val   = val[:50]

write_jsonl(exp2_train, 'jsonl/exp2_train.jsonl')
write_jsonl(exp2_val,   'jsonl/exp2_val.jsonl')

exp2_train_file = upload_jsonl('jsonl/exp2_train.jsonl')
exp2_val_file   = upload_jsonl('jsonl/exp2_val.jsonl')

exp2_job = openai.fine_tuning.jobs.create(
    training_file=exp2_train_file.id,
    validation_file=exp2_val_file.id,
    model='gpt-4.1-nano-2025-04-14',
    seed=42,
    hyperparameters={'n_epochs': 1, 'batch_size': 1},
    suffix='books-bucket200'
)
exp2_job_id = exp2_job.id
print(f'Job started: {exp2_job_id}')
print('Check: https://platform.openai.com/finetune')

In [ ]:
# Run after job finishes
exp2_job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
exp2_model  = get_model_name(exp2_job_id)
evaluate(make_pricer(exp2_model), test)

---
## Experiment 3 — 200 Bucket-Sampled + 3 Epochs

Same data as Experiment 2. Model sees each example 3 times.
Reuses exp2 uploaded files — no re-upload needed.

In [ ]:
exp3_job = openai.fine_tuning.jobs.create(
    training_file=exp2_train_file.id,
    validation_file=exp2_val_file.id,
    model='gpt-4.1-nano-2025-04-14',
    seed=42,
    hyperparameters={'n_epochs': 3, 'batch_size': 1},
    suffix='books-bucket200-3epochs'
)
exp3_job_id = exp3_job.id
print(f'Job started: {exp3_job_id}')
print('Check: https://platform.openai.com/finetune')

In [ ]:
# Run after job finishes
exp3_job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
exp3_model  = get_model_name(exp3_job_id)
evaluate(make_pricer(exp3_model), test)

---
# Final Results

In [ ]:
# Fill in after each evaluate() call above the result here is my own result
results = {
    'XGBoost (ML best)':                  1.55,
    'Baseline (first-100, 1 epoch)':      10.28,
    'Bucket-100 (1 epoch)':               4.31,
    'Bucket-200 (1 epoch)':               37.37,
    'Bucket-200 (3 epochs)':              2.29,
}

best = min(results.values())

print('=' * 58)
print(f'{"Model":<42} {"Error":>10}')
print('=' * 58)
for name, error in results.items():
    flag = '  <- BEST' if error == best else ''
    print(f'  {name:<40} ${error:.2f}{flag}')
print('=' * 58)

#Key findings from my test

1) Bucket sampling cut the error by more than half (baseline $10.28 → Exp 1 $4.31) 
  using the same 100 examples — proving data selection matters more than data quantity
2) 3 epochs with 200 balanced examples ($2.29) came within $0.74 of XGBoost ($1.55)
  which trained on the full dataset
3) The fine-tuned LLM did not beat XGBoost — but it came close using only 200 
  hand-picked training examples vs thousands used by the ML models